# 01 - Lakeflow pipeline: Volume -> bronze -> silver -> gold

Uploads the SQL files in `pipeline/`, creates (or updates) a serverless Lakeflow
Spark Declarative Pipeline, runs it, and prints the evidence: update state,
data quality results from the event log, and the gold table.

In [1]:
import time, yaml
from databricks.connect import DatabricksSession
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.pipelines import FileLibrary, PipelineLibrary
from databricks.sdk.service.workspace import ImportFormat

import logging, warnings
warnings.filterwarnings("ignore")
logging.getLogger("databricks.sdk").setLevel(logging.ERROR)

cfg = yaml.safe_load(open("../config.yml"))
spark = DatabricksSession.builder.profile(cfg["profile"]).serverless(True).getOrCreate()
w = WorkspaceClient(profile=cfg["profile"])
cat, sch = cfg["catalog"], cfg["schema"]

In [2]:
ws_dir = f"/Workspace/Users/{w.current_user.me().user_name}/stress-demo/pipeline"
w.workspace.mkdirs(ws_dir)
files = ["bronze.sql", "silver.sql", "gold.sql"]
for f in files:
    with open(f"../pipeline/{f}", "rb") as fh:
        w.workspace.upload(f"{ws_dir}/{f}", fh, format=ImportFormat.RAW, overwrite=True)

spec = dict(
    name=cfg["pipeline_name"], catalog=cat, schema=sch, serverless=True, development=True,
    configuration={"raw_path": f"/Volumes/{cat}/{sch}/{cfg['volume']}",
                   "mask_fn": f"{cat}.{sch}.mask_counterparty",
                   "llm_endpoint": cfg["llm_endpoint"]},
    libraries=[PipelineLibrary(file=FileLibrary(path=f"{ws_dir}/{f}")) for f in files],
)
found = list(w.pipelines.list_pipelines(filter=f"name LIKE '{cfg['pipeline_name']}'"))
if found:
    pipeline_id = found[0].pipeline_id
    w.pipelines.update(pipeline_id=pipeline_id, **spec)
else:
    pipeline_id = w.pipelines.create(**spec).pipeline_id
print("pipeline_id:", pipeline_id)

pipeline_id: 9c755981-2daf-41ac-93d8-f93c44353e4d


In [3]:
# Full refresh: the raw files are regenerated by notebook 00 on every run, so rebuild all tables from them.
update_id = w.pipelines.start_update(pipeline_id, full_refresh=True).update_id
while (state := w.pipelines.get_update(pipeline_id, update_id).update.state.value) not in ("COMPLETED", "FAILED", "CANCELED"):
    time.sleep(20)
print("update", update_id, "->", state)

update 1beff291-b316-42b1-8f59-fa8b3ef8620d -> COMPLETED


## Data quality: rows passed and dropped by each expectation (from the event log)

In [4]:
spark.sql(f"""
  SELECT e.dataset, e.name AS expectation, e.passed_records, e.failed_records
  FROM event_log('{pipeline_id}'),
  LATERAL explode(from_json(details:flow_progress.data_quality.expectations,
    'array<struct<name:string,dataset:string,passed_records:bigint,failed_records:bigint>>')) AS t(e)
  WHERE event_type = 'flow_progress' AND origin.update_id = '{update_id}'
    AND details:flow_progress.data_quality.expectations IS NOT NULL
  ORDER BY e.dataset, e.name
""").show(truncate=False)

+---------------------------------+---------------+--------------+--------------+
|dataset                          |expectation    |passed_records|failed_records|
+---------------------------------+---------------+--------------+--------------+
|stress_demo.risk.silver_limits   |positive_limit |4             |0             |
|stress_demo.risk.silver_positions|has_sensitivity|201           |1             |
|stress_demo.risk.silver_positions|known_desk     |201           |1             |
|stress_demo.risk.silver_scenarios|has_shock      |32            |0             |
+---------------------------------+---------------+--------------+--------------+



## Row counts per layer

In [5]:
for t in ["bronze_positions", "silver_positions", "bronze_scenarios", "silver_scenarios", "bronze_limits",
          "silver_limits", "gold_stress_pnl", "gold_desk_stress", "gold_commentary"]:
    print(f"{t:18} {spark.table(f'{cat}.{sch}.{t}').count():>5}")

bronze_positions     202


silver_positions     200


bronze_scenarios      32


silver_scenarios      32


bronze_limits          4


silver_limits          4


gold_stress_pnl       32


gold_desk_stress      16


gold_commentary        4


## Gold: stressed P&L by desk (JPY millions)

In [6]:
spark.sql(f"""
  SELECT scenario_id, scenario_name,
    round(sum(CASE WHEN desk = 'Rates'  THEN pnl_jpy END) / 1e6) AS rates,
    round(sum(CASE WHEN desk = 'FX'     THEN pnl_jpy END) / 1e6) AS fx,
    round(sum(CASE WHEN desk = 'Equity' THEN pnl_jpy END) / 1e6) AS equity,
    round(sum(CASE WHEN desk = 'Credit' THEN pnl_jpy END) / 1e6) AS credit,
    round(sum(pnl_jpy) / 1e6) AS total
  FROM {cat}.{sch}.gold_stress_pnl
  GROUP BY ALL ORDER BY scenario_id
""").show(truncate=False)

+-----------+----------------------+------+--------+-------+-------+--------+
|scenario_id|scenario_name         |rates |fx      |equity |credit |total   |
+-----------+----------------------+------+--------+-------+-------+--------+
|S1         |BoJ hikes 100bp       |-293.0|-3670.0 |-1080.0|-282.0 |-5326.0 |
|S2         |Yen surges 15%        |107.0 |-11640.0|-1620.0|-538.0 |-13690.0|
|S3         |Global equity sell-off|326.0 |-6691.0 |-4970.0|-2988.0|-14323.0|
|S4         |Credit spreads widen  |189.0 |-2391.0 |-1988.0|-4901.0|-9091.0 |
+-----------+----------------------+------+--------+-------+-------+--------+



## Gold: desk stress loss vs limit

In [7]:
spark.sql(f"""
  SELECT scenario_name, desk, round(pnl_jpy / 1e9, 1) AS pnl_jpy_bn, round(limit_jpy / 1e9, 1) AS limit_jpy_bn,
    limit_utilization, limit_breach
  FROM {cat}.{sch}.gold_desk_stress ORDER BY scenario_id, desk
""").show(truncate=False)

+----------------------+------+----------+------------+-----------------+------------+
|scenario_name         |desk  |pnl_jpy_bn|limit_jpy_bn|limit_utilization|limit_breach|
+----------------------+------+----------+------------+-----------------+------------+
|BoJ hikes 100bp       |Credit|-0.3      |4.0         |0.071            |false       |
|BoJ hikes 100bp       |Equity|-1.1      |5.0         |0.216            |false       |
|BoJ hikes 100bp       |FX    |-3.7      |8.0         |0.459            |false       |
|BoJ hikes 100bp       |Rates |-0.3      |2.0         |0.147            |false       |
|Yen surges 15%        |Credit|-0.5      |4.0         |0.134            |false       |
|Yen surges 15%        |Equity|-1.6      |5.0         |0.324            |false       |
|Yen surges 15%        |FX    |-11.6     |8.0         |1.455            |true        |
|Yen surges 15%        |Rates |0.1       |2.0         |-0.054           |false       |
|Global equity sell-off|Credit|-3.0      |4

## Gold (GenAI): committee commentary per scenario, and the facts it was grounded on

In [8]:
for r in spark.table(f"{cat}.{sch}.gold_commentary").orderBy("scenario_id").collect():
    print(f"[{r.scenario_id}] {r.scenario_name}\nFACTS: {r.facts}\nCOMMENTARY: {r.commentary}\n")

[S1] BoJ hikes 100bp
FACTS: Scenario: BoJ hikes 100bp. Total stressed P&L: JPY -5.3bn. Desks: Credit: P&L JPY -0.3bn, limit JPY 4.0bn, within limit; Equity: P&L JPY -1.1bn, limit JPY 5.0bn, within limit; FX: P&L JPY -3.7bn, limit JPY 8.0bn, within limit; Rates: P&L JPY -0.3bn, limit JPY 2.0bn, within limit. Top loss drivers: USDJPY (FX desk) JPY -2.4bn; EURJPY (FX desk) JPY -1.3bn.
COMMENTARY: The FX desk drives the stressed loss, with JPY -3.7bn of the JPY -5.3bn total under the 100bp BoJ hike scenario, led by USDJPY (JPY -2.4bn) and EURJPY (JPY -1.3bn). In plain terms, a sharp BoJ hike would likely lift the yen against both the dollar and the euro, so the desk's long USD and long EUR positions against the yen lose value as those pairs fall, and the two exposures compound because both are essentially one yen-strength bet. The committee should note that no limit is breached (FX at JPY -3.7bn against a JPY 8.0bn limit, with Credit, Equity and Rates also within their limits), and should 

## Governance check: counterparty is masked for non `risk_admins`

In [9]:
spark.sql(f"SELECT position_id, desk, counterparty FROM {cat}.{sch}.silver_positions LIMIT 3").show()

+-----------+-----+------------+
|position_id| desk|counterparty|
+-----------+-----+------------+
|      P0001|Rates|    REDACTED|
|      P0002|Rates|    REDACTED|
|      P0003|Rates|    REDACTED|
+-----------+-----+------------+

